In [2]:
!pip install -U bitsandbytes>=0.46.1

## Model Comparison: Base vs. Fine-Tuned

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive
import pandas as pd

# 1. Setup & Mount
drive.mount('/content/drive')
model_name = "EleutherAI/gpt-neo-1.3B"
adapter_path = "/content/drive/MyDrive/alpaca-portfolio-project/checkpoint-900" # Update to your latest CP

# 2. Load Configs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. Load Models
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load Base
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)

# Load Fine-Tuned (This wraps the base model)
ft_model = PeftModel.from_pretrained(base_model, adapter_path)

# 4. Define Comparison Prompts
test_prompts = [
    "Write a short poem about a lonely robot.",
    "Explain how a microwave works to a 5-year-old.",
    "Give me a list of 3 healthy breakfast ideas.",
    "Write a formal email to a boss asking for a day off."
]

def generate_output(model, prompt, is_alpaca=True):
    if is_alpaca:
        # Alpaca Format
        formatted_prompt = (
            "Below is an instruction that describes a task. "
            f"Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
    else:
        # Base model just gets the raw prompt
        formatted_prompt = prompt

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if is_alpaca:
        return decoded.split("### Response:")[-1].strip()
    return decoded.strip()

# 5. Run Comparison
results = []

for p in test_prompts:
    print(f"Processing: {p}")

    # Get Base Response (using the disable_adapter context)
    with ft_model.disable_adapter():
        base_resp = generate_output(ft_model, p, is_alpaca=False)

    # Get Fine-Tuned Response
    ft_resp = generate_output(ft_model, p, is_alpaca=True)

    results.append({
        "Instruction": p,
        "Base GPT-Neo 1.3B": base_resp,
        "Fine-Tuned (Alpaca-Neo)": ft_resp
    })

# 6. Display as Table
df = pd.DataFrame(results)
# Set column width so we can actually read the responses in Colab
pd.set_option('display.max_colwidth', 500)
display(df)

# Optional: Save to CSV for your GitHub
df.to_csv("model_comparison.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-1.3B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...23}.attn.attention.masked_bias | UNEXPECTED |  | 
transformer.h.{0...22}.attn.attention.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing: Write a short poem about a lonely robot.
Processing: Explain how a microwave works to a 5-year-old.
Processing: Give me a list of 3 healthy breakfast ideas.
Processing: Write a formal email to a boss asking for a day off.


,Instruction,Base GPT-Neo 1.3B,Fine-Tuned (Alpaca-Neo)
0,Write a short poem about a lonely robot.,"Write a short poem about a lonely robot.\n\nThe Robot:\n\nA robot is a robot, unless it’s a person… in which case, it‘s not a robot. A robot is one who is not a person.\nYou can’t build a robot that can think!\n\nBut you can make a robot as if it were a person, so it can walk and talk and do all the things that a human would do.\nIt can learn! It can listen!\nIt has","I walk in a lonely world,\nI hear my footsteps fading away.\nI'm alone, but I know that I'm not alone.\nThere are things I can do,\nBut I can't really make them happen.\nThe words seem to tumble on the page,\nAnd they never seem to fit together.\nA lonely robot, I'm becoming a friend.\nNow I can just be with you,\nTake a walk, and watch you grow.\nThis lonely world"
1,Explain how a microwave works to a 5-year-old.,"Explain how a microwave works to a 5-year-old.\n\nDale Gaskins, a fifth-grader at St. John’s Elementary School in Derry, Conn., has a microwave that he uses to get hot food ready for school. He likes to cook some things himself, and he tells his mother not to make him do it.\nGaskins was born in Dutchess County, N.Y., but he didn’t grow up in the area. His father, a construction worker, moved to","A microwave is a device that heats up food and can be used to heat and heat up food without smoking hot food. A microwave uses microwaves to heat food, which are radio frequency waves that heat up the food and cause it to cook. The microwave is usually used to cook food for children because it is safe and easy to use. It can also be used for science experiments, cooking frozen foods, and other tasks. To use a microwave, you need to have a microwave oven, which is"
2,Give me a list of 3 healthy breakfast ideas.,"Give me a list of 3 healthy breakfast ideas.\n\nYou may have heard of the ‘Diet for Diabetics’, but did you know that the majority of people with diabetes are overweight and obese? So, what's the point of having a diet to lose weight when you'll put on weight anyway? Well, this is the answer.\nThe problem with dieting is that it's difficult to maintain.\nSo, what would happen if there was a way to use the calorie burning potential of exercise to help you lose","1. Fruit Juice with Berry Smoothie:\nFruit juices are convenient and nutritious and are a great way to start your day. This smoothie is packed with fruit, protein and complex carbohydrates to help you stay active. \n2. Avocado Toast:\nAvocados are delicious and healthy, and a perfect breakfast option. Try this creamy avocado toast with a drizzle of honey and a side of blueberries. \n3. Eggs Benedict:\nEggs Benedict is"
3,Write a formal email to a boss asking for a day off.,"Write a formal email to a boss asking for a day off.\n\nThis is the kind of request that has been coming in from the office for years, so it should be a familiar scenario.\nIf your boss has decided to take a day-off, you can still get out of the office and meet up with a friend, but you might have to wait until Monday or Tuesday to do so.\nThe best plan is to make a formal request on Friday morning when you’re usually available, and then ask for a short break.\nThat","Dear Sir or Madam,\n\nI am writing to ask you to please give me a day of your time off today. I know this is a difficult request, but I truly need it. I am in need of a day to myself to reflect and recharge my batteries. Please give me your permission to use this day to reflect on the past year-and-a-half and perhaps consider a career change. Thank you for your consideration and I look forward to hearing from you soon."


Looking at the result of both models, we can easily notice how the fine tuned model have perfectly captured the "Value Proposition" of fine-tuning.
Here is a breakdown of what these results actually prove:
<ol>
<li>
 Instruction Adherence (The Biggest Win):
 <ul>
<li>
Base Model: It completely fails to understand it's an assistant. In Row 1 (Microwave) and Row 2 (Breakfast), it just starts writing random blog posts or news stories about children in Connecticut.

<li>
Fine-tunned Model: It recognizes the Instruction/Response pattern every single time. It provides a poem, a list, and an email exactly as requested.
</ul>
<li>

Format Recognition:
<ul>
<li>

In the Breakfast Ideas prompt, the Fine-tunned model successfully used a numbered list format (1. Fruit Juice... 2. Avocado Toast...). The base model just ranted about diabetes. This proves the LoRA layers successfully "re-wired" the model's output structure.
</ul>
<li>

The "Small Model" Quirk (Honest Documentation):
<ul>
<li>
Observation: In the Microwave explanation, your model still struggles with "circular logic" ("heats up food... to heat and heat up food").
<li>
Why? This is a great moment to explain that a 1.3B parameter model has limited reasoning. It learned how to answer, but it still lacks the deep "common sense" of a 70B model.